# F-09 Whisper 5차 학습 — final2 이어받기 + 올바른 v2 데이터

**변경 사항 (finetune_4th.ipynb 대비)**
- 출발점: `final2` (9.6% CER) — final3/4 오염 가중치 버리고 재시작
- 데이터: `senior_speech_v2` (SP 태그 수정 완료된 버전)
- lr: `1e-5` — final2 이어받기 기준
- 평가: Trainer 내부 CER 미사용, 학습 완료 후 셀 04에서 직접 generate 호출로만 평가

**실행 순서**: 셀 01 → 02 → 03 → 04

In [ ]:
# 셀 01 — 라이브러리 설치 + Drive 마운트 + 경로 설정
!pip install -q \
    transformers \
    datasets \
    peft \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    tensorboard \
    "torchao>=0.16.0"

from google.colab import drive
from pathlib import Path
import os

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT      = Path('/content/drive/MyDrive/Dadam_dataSet')
DATASET_PATH    = DRIVE_ROOT / 'processed/senior_speech_v2'
CHECKPOINT_DIR  = DRIVE_ROOT / 'checkpoints/whisper-senior'
FINAL2_DIR      = CHECKPOINT_DIR / 'final2'   # 출발점 (9.6% CER)
FINAL5_DIR      = CHECKPOINT_DIR / 'final5'   # 5차 학습 결과
CHECKPOINT5_DIR = CHECKPOINT_DIR / 'stage5'
CHECKPOINT5_DIR.mkdir(parents=True, exist_ok=True)

print('완료')
print(f'데이터셋 경로 : {DATASET_PATH}')
print(f'출발점       : {FINAL2_DIR}  존재={FINAL2_DIR.exists()}')
print(f'저장 위치    : {FINAL5_DIR}')

In [ ]:
# 셀 02 — v2 데이터셋 확인 (SP 태그 수정 여부 검증)
from datasets import load_from_disk
import re

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'v2 데이터셋 없음: {DATASET_PATH}\n셀 02(finetune_4th.ipynb)에서 재전처리 먼저 실행')

texts = load_from_disk(str(DATASET_PATH))['train'].select_columns(['text'])
sp_pattern = re.compile(r'\(SP[: ][^)]+\)')

found = False
for row in texts.select(range(500)):
    if sp_pattern.search(row['text']):
        print('SP 태그 발견 — 재전처리 필요:', row['text'])
        found = True
        break

if not found:
    print('SP 태그 없음 — v2 데이터셋 정상')
    print(f'train: {len(texts)}개')

In [ ]:
# 셀 03 — 5차 학습 (final2 이어받기)
# 백그라운드 실행 활성화 후 실행할 것 (런타임 > 백그라운드 실행)
import os
import re
import torch
from dataclasses import dataclass
from typing import Any
from datasets import load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import PeftModel

MODEL_ID = 'openai/whisper-large-v3-turbo'

if not FINAL2_DIR.exists():
    raise FileNotFoundError(f'final2 없음: {FINAL2_DIR}')

processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')

base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
base_model.generation_config.language           = 'korean'
base_model.generation_config.task               = 'transcribe'
base_model.generation_config.forced_decoder_ids = None

model = PeftModel.from_pretrained(base_model, str(FINAL2_DIR), is_trainable=True)
model.print_trainable_parameters()

dataset = load_from_disk(str(DATASET_PATH))
print(dataset)

# ── DataCollator ─────────────────────────────────────────────────────────────
@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        labels = self.processor.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=448,
        ).input_ids
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'input_features': inputs.input_features, 'labels': labels}

# ── 학습 설정 — compute_metrics 제거, eval은 loss만 모니터링 ──────────────────
# Trainer 내부 generate CER이 실제와 달랐던 문제 회피
# 최종 평가는 셀 04에서 직접 generate 호출로 수행
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT5_DIR),
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,         # 유효 배치: 64
    learning_rate=1e-5,                    # final2 이어받기 — 2차와 동일 lr
    warmup_steps=100,
    max_steps=2000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='loss',          # loss 기준 best 선택 — CER 오작동 회피
    greater_is_better=False,
    predict_with_generate=False,           # Trainer 내부 generate 비활성화
    report_to='tensorboard',
    save_total_limit=3,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'].select(range(500)),
    data_collator=WhisperDataCollator(processor=processor),
    processing_class=processor.feature_extractor,
)

# 체크포인트 이어받기
last_checkpoint = None
checkpoints = sorted(CHECKPOINT5_DIR.glob('checkpoint-*'), key=os.path.getmtime)
if checkpoints:
    last_checkpoint = str(checkpoints[-1])
    print(f'체크포인트 발견 — 이어서 학습: {last_checkpoint}')
else:
    print('체크포인트 없음 — 처음부터 5차 학습 시작')

trainer.train(resume_from_checkpoint=last_checkpoint)

model.save_pretrained(str(FINAL5_DIR))
processor.save_pretrained(str(FINAL5_DIR))
print(f'5차 학습 완료. 저장 위치: {FINAL5_DIR}')

In [ ]:
# 셀 04 — final5 평가 (직접 generate 호출 — Trainer 내부 CER 미사용)
!pip install -q evaluate jiwer

import re
import torch
import evaluate
from torch.utils.data import DataLoader
from dataclasses import dataclass
from typing import Any
from pathlib import Path

if 'FINAL5_DIR' not in dir():
    DRIVE_ROOT     = Path('/content/drive/MyDrive/Dadam_dataSet')
    DATASET_PATH   = DRIVE_ROOT / 'processed/senior_speech_v2'
    CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints/whisper-senior'
    FINAL5_DIR     = CHECKPOINT_DIR / 'final5'

PUNCT_PATTERN = re.compile(r'[.?!,。、]')

def clean_text(text: str) -> str:
    return PUNCT_PATTERN.sub('', text).replace(' ', '').strip()

cer_metric   = evaluate.load('cer')
EVAL_SAMPLES = 500
GEN_KWARGS   = dict(language='korean', task='transcribe', num_beams=1)

if 'model' not in dir() or not hasattr(model, 'generate'):
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    from peft import PeftModel
    from datasets import load_from_disk
    MODEL_ID   = 'openai/whisper-large-v3-turbo'
    processor  = WhisperProcessor.from_pretrained(MODEL_ID, language='Korean', task='transcribe')
    base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    base_model.generation_config.language           = 'korean'
    base_model.generation_config.task               = 'transcribe'
    base_model.generation_config.forced_decoder_ids = None
    model = PeftModel.from_pretrained(base_model, str(FINAL5_DIR), is_trainable=False)
    model = model.half().to('cuda').eval()
    dataset = load_from_disk(str(DATASET_PATH))

@dataclass
class EvalCollator:
    processor: Any

    def __call__(self, features):
        audio_arrays = [f['audio']['array'] for f in features]
        texts        = [f['text'] for f in features]
        inputs = self.processor(
            audio_arrays,
            sampling_rate=16_000,
            return_tensors='pt',
            padding='max_length',
            max_length=480_000,
            truncation=True,
        )
        return {'input_features': inputs.input_features, 'texts': texts}

eval_subset = dataset['validation'].select(range(EVAL_SAMPLES))
loader      = DataLoader(eval_subset, batch_size=16, collate_fn=EvalCollator(processor))

all_preds, all_refs = [], []

for batch in loader:
    input_feats = batch['input_features'].to('cuda', dtype=torch.float16)
    with torch.no_grad():
        pred_ids = model.generate(input_features=input_feats, **GEN_KWARGS)
    preds = processor.batch_decode(pred_ids, skip_special_tokens=True)
    all_preds.extend([clean_text(p) for p in preds])
    all_refs.extend( [clean_text(r) for r in batch['texts']])

cer_result = cer_metric.compute(predictions=all_preds, references=all_refs)

print(f'final5 CER ({EVAL_SAMPLES}개): {cer_result:.4f}  →  {cer_result * 100:.2f}%')
print()
print('── 예측 vs 정답 샘플 5개 ──')
for i in range(5):
    print(f'  정답: {all_refs[i]}')
    print(f'  예측: {all_preds[i]}')
    print()

In [ ]:
# 셀 05 — v2 validation 레이블 진단 (SP 태그 잔재 + 샘플 확인)
from datasets import load_from_disk
from pathlib import Path
import re

if 'DATASET_PATH' not in dir():
    DATASET_PATH = Path('/content/drive/MyDrive/Dadam_dataSet/processed/senior_speech_v2')

texts = load_from_disk(str(DATASET_PATH))['validation'].select_columns(['text'])

sp_pattern = re.compile(r'\(SP[: ][^)]+\)')
sp_found = 0

for row in texts.select(range(1000)):
    t = row['text']
    if sp_pattern.search(t):
        sp_found += 1
        if sp_found <= 5:
            print('SP 태그 발견:', t)

print('
검사: 1000개, SP 태그 잔재: ' + str(sp_found) + '개')
print('
--- validation 첫 5개 레이블 ---')
for row in texts.select(range(5)):
    print(repr(row['text']))
